[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/15-projects/p10-finetune-lora-dpo/colab_gpu_trl-colab-only.ipynb)

# P10 - LoRA SFT + DPO with TRL on a free T4 GPU (colab-only)
**Runtime -> Change runtime type -> T4 GPU** before running. This uses Hugging Face TRL's `SFTTrainer` and `DPOTrainer` on SmolLM2-360M-Instruct (change `MODEL` to try e.g. `Qwen/Qwen2.5-0.5B-Instruct`). Expect roughly 10-20 minutes in total.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code and install the pinned packages
We keep Colab's own CUDA build of PyTorch.

In [ ]:
import os, pathlib, subprocess, sys
PROJECT = "p10-finetune-lora-dpo"
if not pathlib.Path("src").is_dir():
    if not pathlib.Path("AI-Engineer").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vvvvvivekkk/AI-Engineer.git"], check=True)
    os.chdir(f"AI-Engineer/15-projects/{PROJECT}")
reqs = [l for l in open("requirements.txt") if l.strip() and not l.startswith(("torch==", "#"))]
open("/tmp/req.txt", "w").writelines(reqs)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req.txt"], check=True)
print(os.getcwd())

## 3. Build the datasets

In [ ]:
!python -m src.make_data

## 4. Baseline: prompting the base model
How often does the untuned model return valid, exactly correct JSON - with and without one example in the prompt?

In [ ]:
import torch
from src.common import evaluate, load_base, load_jsonl, DATA
MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
test = load_jsonl(DATA / "sft_test.jsonl")[:100]
tok, base = load_base(MODEL)
base = base.to("cuda")
print("zero-shot:", evaluate(base, tok, test)[0])
print("few-shot: ", evaluate(base, tok, test, few_shot=True)[0])
del base; torch.cuda.empty_cache()

## 5. SFT + DPO with TRL
`src/trl_train.py`: SFT with LoRA on all linear layers and loss only on the answer (`completion_only_loss=True`), merge, then DPO with a new LoRA adapter (`peft_config`), so the frozen reference is the SFT model. `fp16=True` because the T4 has no bfloat16.

In [ ]:
!python -m src.trl_train --model HuggingFaceTB/SmolLM2-360M-Instruct --gpu --sft-steps 300 --dpo-steps 100 --batch 8 --eval-n 100

## 6. Results

In [ ]:
print(open('outputs/trl_results.json').read())

## 7. Merge the DPO adapter and export
Merge on top of the SFT-merged model, save, and convert to GGUF with llama.cpp so Ollama can serve it.

In [ ]:
from peft import PeftModel
from src.trl_train import merged_sft_model
tok, sft = merged_sft_model(MODEL, "outputs/trl-sft-adapter", gpu=False)
final = PeftModel.from_pretrained(sft, "outputs/trl-dpo-adapter").merge_and_unload()
final.save_pretrained("outputs/final-merged"); tok.save_pretrained("outputs/final-merged")
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt
!python /content/llama.cpp/convert_hf_to_gguf.py outputs/final-merged --outfile outputs/ticket-extractor-q8_0.gguf --outtype q8_0
!ls -lh outputs/*.gguf

## 8. Download
Download `outputs/ticket-extractor-q8_0.gguf`, put it next to the `Modelfile` that `python -m src.merge_export` writes, and run `ollama create ticket-extractor -f Modelfile`.

In [ ]:
from google.colab import files
files.download("outputs/ticket-extractor-q8_0.gguf")